# 03 - Canny dataset + leak-free split (MuralDH Mural512)
CPU is enough (no GPU needed). Input: the MuralDH dataset. Output (in `/kaggle/working/muraldh_canny`):
- `canny/<name>.png` - Canny map (100/200, same as the project's `canny_map`) of every crop
- `manifest.csv` - name, source id, crop position, split
- `splits.json` - source ids per split

**Why split by source id:** file names are `img_<N>crop_<i>_<j>.png`: four crops (2x2) of one source mural N. Crops of the same mural are near-duplicates, so a random split by crop leaks. Splitting by N keeps all four crops of a mural in the same split.

After it finishes: *Save Version -> Save & Run All*, then on the Output tab choose **New Dataset** and name it `MuralDH-Canny`.

In [ ]:
import re, json, random, cv2, numpy as np, pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

IMAGE_DIR = Path("/kaggle/input/datasets/balamsd07/muraldh/Mural512/Mural512")
OUT = Path("/kaggle/working/muraldh_canny"); (OUT/"canny").mkdir(parents=True, exist_ok=True)
CANNY_LOW, CANNY_HIGH = 100, 200          # must match config.CANNY_LOW/HIGH in the project
SEED = 42
FRAC_VAL, FRAC_TEST = 0.05, 0.10          # of SOURCE murals

pat = re.compile(r"img_(\d+)crop_(\d+)_(\d+)\.png$")
rows, bad = [], []
for p in sorted(IMAGE_DIR.glob("*.png")):
    m = pat.match(p.name)
    if not m: bad.append(p.name); continue
    rows.append(dict(name=p.name, src=int(m[1]), ci=int(m[2]), cj=int(m[3])))
df = pd.DataFrame(rows)
print("crops:", len(df), "| sources:", df.src.nunique(), "| unmatched names:", len(bad), bad[:5])
print(df.groupby("src").size().value_counts().sort_index().to_dict(), "<- crops per source (count of sources)")

In [ ]:
srcs = sorted(df.src.unique()); random.Random(SEED).shuffle(srcs)
n_test, n_val = int(len(srcs)*FRAC_TEST), int(len(srcs)*FRAC_VAL)
split_of = {}
for s in srcs[:n_test]: split_of[s] = "test"
for s in srcs[n_test:n_test+n_val]: split_of[s] = "val"
for s in srcs[n_test+n_val:]: split_of[s] = "train"
df["split"] = df.src.map(split_of)
print(df.split.value_counts().to_dict())
assert not (set(df[df.split=="train"].src) & set(df[df.split=="test"].src)), "leak!"
json.dump({k: sorted(int(s) for s,v in split_of.items() if v==k) for k in ("train","val","test")}, open(OUT/"splits.json","w"))
df.to_csv(OUT/"manifest.csv", index=False)

In [ ]:
def canny_of(path):
    gray = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2GRAY)     # same as canny_map: RGB->GRAY then Canny
    return cv2.Canny(gray, CANNY_LOW, CANNY_HIGH)

dens = []
for name in tqdm(df.name):
    e = canny_of(IMAGE_DIR/name)
    cv2.imwrite(str(OUT/"canny"/name), e)
    dens.append(float((e > 0).mean()*100))
df["edge_pct"] = dens; df.to_csv(OUT/"manifest.csv", index=False)
print(df.edge_pct.describe().round(2).to_dict())
print("files:", len(list((OUT/'canny').glob('*.png'))))

In [ ]:
import matplotlib.pyplot as plt
pick = df.sample(4, random_state=1).name.tolist()
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for i, n in enumerate(pick):
    ax[0,i].imshow(cv2.cvtColor(cv2.imread(str(IMAGE_DIR/n)), cv2.COLOR_BGR2RGB)); ax[0,i].set_title(n, fontsize=8)
    ax[1,i].imshow(cv2.imread(str(OUT/"canny"/n), 0), cmap="gray")
    ax[0,i].axis("off"); ax[1,i].axis("off")
plt.tight_layout(); plt.savefig(OUT/"canny_examples.png", dpi=120); plt.show()
print("Done. Now save a version and publish /kaggle/working/muraldh_canny as dataset 'MuralDH-Canny'.")